In [2]:
import torch
device = torch.device('cuda:0')
import pandas as pd
from go_ml.train_utils import get_enzyme_df, enzyme_iterator
import transformers
tokenizer = transformers.AutoTokenizer.from_pretrained('facebook/esm2_t6_8M_UR50D')
enzyme_df = get_enzyme_df()
enzyme_l = list(enzyme_iterator(enzyme_df, tokenizer))

In [21]:
import pickle
with open('notebook_cache/test_100_bert_distr.pkl', 'rb') as f:
    bert_distr_l = pickle.load(f)

with open('notebook_cache/test_1000_mut_eval.pkl', 'rb') as f:
    mut_seq_l, logit_preds_csr = pickle.load(f)

In [27]:
import json
train_path = "/home/andrew/cafa5_team/data/"
with open(f"{train_path}/cafa_dataset/go_terms.json", "r") as f:
    go_terms = json.load(f)
term_ind_map = {t:i for i, t in enumerate(go_terms)}
import goatools
from goatools.obo_parser import GODag
godag = GODag('../go-basic.obo')
from goatools.godag.go_tasks import get_go2parents
optional_relationships = set()
go2parents_isa = get_go2parents(godag, optional_relationships)

def get_parent_inds(go_ind):
    go_term = go_terms[go_ind]
    if(not go_term in go2parents_isa):
        return go_ind    
    parents = go2parents_isa[go_term]
    parent_inds = [term_ind_map[p] for p in parents if p in term_ind_map]
    return parent_inds

../go-basic.obo: fmt(1.2) rel(2024-04-24) 45,667 Terms


In [35]:
go2parents_isa['GO:0098916']

{'GO:0099537'}

In [41]:
def get_ancestor_terms(go_term):
    seen = set()
    to_visit = [go_term]
    while to_visit:
        curr_go = to_visit.pop()
        if(curr_go in seen):
            continue
        seen.add(curr_go)
        if(curr_go in go2parents_isa):
            parents = go2parents_isa[curr_go]
            to_visit.extend(parents)
    seen.add(go_term)
    return list(seen)

def get_ancestor_ind(go_ind):
    return  [term_ind_map[p] for p in get_ancestor_terms(go_terms[go_ind]) if p in term_ind_map]

[15783, 9826, 103, 100, 23692]

In [42]:
eval_logits_l = []
for i, test_prot in enumerate(enzyme_l[:100]):
    go_ind = test_prot['go_ind']
    ancestor_ind = get_ancestor_ind(go_ind)
    eval_logits = logit_preds_csr[i*1000:i*1000+1000].todense()[:, ancestor_ind]
    print(eval_logits.shape)

(1000, 5)
(1000, 6)
(1000, 6)
(1000, 5)
(1000, 7)
(1000, 8)
(1000, 6)
(1000, 7)
(1000, 8)
(1000, 6)
(1000, 4)
(1000, 8)
(1000, 5)
(1000, 6)
(1000, 7)
(1000, 6)
(1000, 5)
(1000, 8)
(1000, 4)
(1000, 6)
(1000, 5)
(1000, 8)
(1000, 6)
(1000, 8)
(1000, 8)
(1000, 6)
(1000, 7)
(1000, 13)
(1000, 5)
(1000, 3)
(1000, 13)
(1000, 6)
(1000, 6)
(1000, 6)
(1000, 6)
(1000, 8)
(1000, 6)
(1000, 7)
(1000, 7)
(1000, 7)


KeyboardInterrupt: 

In [16]:
aa_str = 'LAGVSERTIDPKQNFYMHWC'
aa_ind = [tokenizer.get_vocab()[c] for c in aa_str]
base_logit_l = []
for i in range(100):
    test_prot = enzyme_l[i]
    sl = len(test_prot['seq'])
    bert_distr = bert_distr_l[i]
    bert_distr = bert_distr[1:sl+1, aa_ind]
    bert_distr /= bert_distr.sum(dim=1, keepdim=True)
    bert_logits = torch.log(bert_distr)
    base_logit_l.append(bert_logits)

In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SeqFBayes(nn.Module):
    def __init__(self, base_logits):
        super().__init__()
        self.pos_logit = torch.nn.Parameter(base_logits.clone())
        self.neg_logit = torch.nn.Parameter(base_logits.clone())
        self.base_logit = base_logits
        self.logit_prior = torch.nn.Parameter(torch.tensor([0.0, 0.0]))
    
    def forward(self, seq):
        log_prior = F.log_softmax(self.logit_prior, dim=0)
        ln, v = self.base_logit.shape
        pos_logprob = torch.log_softmax(self.pos_logit, dim=1)[torch.arange(ln).reshape(1, -1), seq].sum(dim=1)
        neg_logprob = torch.log_softmax(self.neg_logit, dim=1)[torch.arange(ln).reshape(1, -1), seq].sum(dim=1)
        joint_logprob = torch.stack([pos_logprob + log_prior[0], neg_logprob + log_prior[1]], dim=-1)
        base_logprob = torch.logsumexp(joint_logprob, dim=-1)
        pos_cond_prob = pos_logprob + log_prior[0] - base_logprob
        neg_cond_prob = neg_logprob + log_prior[1] - base_logprob
        return torch.stack([neg_cond_prob, pos_cond_prob], dim=-1)
    
def to_seqf_tok(tok, sl):
    return tok[:, 1:sl+1] - 4 #-4 to convert to pure aa tokens from BERT tokens

from go_ml.data_utils import ProtDataset, get_seq_collator
from torch.utils.data import DataLoader
collate_seqs = get_seq_collator(tokenizer, max_length=1024, add_special_tokens=True)

def fit_seqf(base_logits, mut_seq, eval_logits):
    mut_ds = ProtDataset(['csatest']*len(mut_seq), [x[0] for x in mut_seq])
    mut_dl = DataLoader(mut_ds, shuffle=False, batch_size=60, collate_fn=collate_seqs)
    mut_seq_tok = torch.cat([batch['seq_ind'] for batch in mut_dl])
    mut_seq_tok = to_seqf_tok(mut_seq_tok, sl).to(device)
    joint_logits = torch.stack([torch.zeros_like(eval_logits), eval_logits], dim=-1)
    mut_seq_target_log = torch.log_softmax(joint_logits, dim=-1)

    model = SeqFBayes(base_logits).to(device)
    mut_seq_tok = mut_seq_tok.to(device)
    mut_seq_target_log = mut_seq_target_log.to(device)
    optimizer = torch.optim.Adam(params=model.parameters(), lr=3e-4)
    
    for epoch in range(500):
        logprob = model.forward(mut_seq_tok) # N x C
        # print(logprob.shape)
        # seq_loss = F.kl_div(logprob, mut_seq_target_log, log_target=True, reduction='batchmean')
        seq_loss = F.kl_div(logprob, mut_seq_target_log, log_target=True, reduction='batchmean')

        reg_loss = (F.kl_div(torch.log_softmax(model.pos_logit, dim=1), 
                        torch.log_softmax(model.base_logit, dim=1), 
                        log_target=True, reduction='sum') + 
                        F.kl_div(torch.log_softmax(model.neg_logit, dim=1), 
                        torch.log_softmax(model.base_logit, dim=1), 
                        log_target=True, reduction='sum')
                        )
        loss = seq_loss + 1e-1*reg_loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return model

for prot_data, base_logits, mut_seq in zip(enzyme_l, base_logit_l, mut_seq_l)